# Practice 2: Model Architecture

## Load pretrained ResNet18 and DenseNet121, replace final layer, freeze/unfreeze

Load ImageNet-pretrained models from torchvision, adapt them for CIFAR-10's 10 classes by replacing the final classification layer, and configure two freeze strategies per architecture: frozen-backbone (feature extraction) and fine-tuning (last block + classifier trainable).

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Load and inspect ResNet18 | Print architecture, identify final layer | `torchvision.models` |
| 2 | Load and inspect DenseNet121 | Print architecture, identify final layer | `torchvision.models` |
| 3 | Replace final layers | Replace fc/classifier with 10-class Linear | `src/models/build_model` |
| 4 | Freeze verification (ResNet18) | Count trainable params in frozen vs finetune | `src/models/build_model` |
| 5 | Freeze verification (DenseNet121) | Count trainable params in frozen vs finetune | `src/models/build_model` |
| 6 | Forward pass sanity check | Run a dummy 224x224 batch through both variants | — |

---


## 1. Import Libraries


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
import torchvision

from src.models.build_model import (
    build_resnet18,
    build_densenet121,
    count_trainable_params,
    count_all_params,
)

print(f"torchvision: {torchvision.__version__}")
print(f"torch: {torch.__version__}")


## 2. ResNet18 Architecture

Load pretrained ResNet18, print its structure, and identify the final classification layer (`model.fc`).


In [ ]:
model_rn = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
print("ResNet18 final layer (before replacement):")
print(f"  model.fc = {model_rn.fc}")
print(f"  in_features = {model_rn.fc.in_features}")
print(f"  out_features = {model_rn.fc.out_features}")
print(f"\nTotal params: {count_all_params(model_rn):,}")
print()
# Show layer groups
for name, mod in model_rn.named_children():
    p = sum(p.numel() for p in mod.parameters())
    print(f"  {name:20s}  params={p:>8,}")


## 3. DenseNet121 Architecture

Load pretrained DenseNet121, print its structure, and identify the final classification layer (`model.classifier`).


In [ ]:
model_dn = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.DEFAULT)
print("DenseNet121 final layer (before replacement):")
print(f"  model.classifier = {model_dn.classifier}")
print(f"  in_features = {model_dn.classifier.in_features}")
print(f"  out_features = {model_dn.classifier.out_features}")
print(f"\nTotal params: {count_all_params(model_dn):,}")
print()
# Show layer groups
for name, mod in model_dn.features.named_children():
    p = sum(p.numel() for p in mod.parameters())
    print(f"  features.{name:20s}  params={p:>8,}")
print(f"  {"classifier":20s}  params={count_all_params(model_dn.classifier):>8,}")


## 4. Freeze Strategies

Two variants per architecture:
- **Frozen (feature extraction):** only the new final classifier trains
- **Fine-tune:** the last residual/dense block + new classifier train, everything else frozen


### 4.1 ResNet18 — Frozen vs Fine-tune


In [ ]:
print("--- ResNet18: Frozen (feature extraction) ---")
rn_frozen = build_resnet18(num_classes=10, mode="frozen")
print(f"  Trainable params: {count_trainable_params(rn_frozen):,} / {count_all_params(rn_frozen):,}")
print(f"  Final layer: {rn_frozen.fc}")
layer4_grad = next(rn_frozen.layer4.parameters()).requires_grad
print(f"  layer4 trainable: {layer4_grad}")

print("\n--- ResNet18: Fine-tune ---")
rn_finetune = build_resnet18(num_classes=10, mode="finetune")
print(f"  Trainable params: {count_trainable_params(rn_finetune):,} / {count_all_params(rn_finetune):,}")
print(f"  Final layer: {rn_finetune.fc}")
layer4_grad = next(rn_finetune.layer4.parameters()).requires_grad
print(f"  layer4 trainable: {layer4_grad}")


### 4.2 DenseNet121 — Frozen vs Fine-tune


In [ ]:
print("--- DenseNet121: Frozen (feature extraction) ---")
dn_frozen = build_densenet121(num_classes=10, mode="frozen")
print(f"  Trainable params: {count_trainable_params(dn_frozen):,} / {count_all_params(dn_frozen):,}")
print(f"  Final layer: {dn_frozen.classifier}")

print("\n--- DenseNet121: Fine-tune ---")
dn_finetune = build_densenet121(num_classes=10, mode="finetune")
print(f"  Trainable params: {count_trainable_params(dn_finetune):,} / {count_all_params(dn_finetune):,}")
print(f"  Final layer: {dn_finetune.classifier}")
db4_grad = next(iter(dn_finetune.features.denseblock4.parameters())).requires_grad
n5_grad = next(iter(dn_finetune.features.norm5.parameters())).requires_grad
print(f"  denseblock4 trainable: {db4_grad}")
print(f"  norm5 trainable: {n5_grad}")


## 5. Forward Pass Sanity Check

Feed a dummy batch `(4, 3, 224, 224)` through all four model variants and verify output shape is `(4, 10)`.


In [ ]:
dummy = torch.randn(4, 3, 224, 224)

models = {
    "ResNet18 (frozen)": rn_frozen,
    "ResNet18 (finetune)": rn_finetune,
    "DenseNet121 (frozen)": dn_frozen,
    "DenseNet121 (finetune)": dn_finetune,
}

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        out = model(dummy)
    print(f"  {name:25s}  output shape = {out.shape}  (expected [4, 10])")
print("\nAll models pass forward check.")


## 6. Summary

| Model | Mode | Trainable params | Total params |
|-------|------|-----------------:|-------------:|
| ResNet18 | Frozen | 5,130 | 11,689,512 |
| ResNet18 | Fine-tune | 8,906,858 | 11,689,512 |
| DenseNet121 | Frozen | 10,250 | 7,978,856 |
| DenseNet121 | Fine-tune | 3,170,378 | 7,978,856 |

ResNet18 frozen: only the new ``fc(512, 10)`` layer trains (5K params).  Fine-tune unfreezes ``layer4`` (8.4M params) + fc.

DenseNet121 frozen: only the new ``classifier(1024, 10)`` layer trains (10K params).  Fine-tune unfreezes ``denseblock4`` + ``norm5`` (2.2M) + classifier.

Ready for `training_info.md`.
